# Prédiction du Prix de Vente d'une Voiture d'Occasion
## Notebook 2 : Pré-traitement des Données (Preprocessing)
**EHTP — MSDE Édition 7 | Module 5 : Machine Learning**
---

## I. Chargement des Librairies et du Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/autos.csv', encoding='latin-1')
print(f"Dataset chargé : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head(3)

Dataset chargé : 371,528 lignes x 21 colonnes


,index,dateCrawled,name,seller,offerType,price,abtest,vehicleType,yearOfRegistration,gearbox,powerPS,model,kilometer,monthOfRegistration,fuelType,brand,notRepairedDamage,dateCreated,nrOfPictures,postalCode,lastSeen
0,0,2016-03-24 11:52:17,Golf_3_1.6,privat,Angebot,480,test,NaN,1993,manuell,0,golf,150000,0,benzin,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,1,2016-03-24 10:58:45,A5_Sportback_2.7_Tdi,privat,Angebot,18300,test,coupe,2011,manuell,190,NaN,125000,5,diesel,audi,ja,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2,2016-03-14 12:52:21,"Jeep_Grand_Cherokee_""Overland""",privat,Angebot,9800,test,suv,2004,automatik,163,grand,125000,8,diesel,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46


### Interprétation

Nous repartons du fichier brut `autos.csv` (le même que dans le notebook d'EDA) plutôt que d'un export intermédiaire, afin que ce notebook de preprocessing soit autonome et reproductible.

L'option `encoding='latin-1'` est nécessaire car ce dataset (annonces allemandes) contient des caractères accentués qui ne sont pas en UTF-8.

À ce stade, le dataset contient encore toutes les colonnes brutes, y compris celles identifiées dans l'EDA comme non informatives (`index`, `dateCrawled`, `nrOfPictures`, `postalCode`, etc.). L'étape suivante consiste à les retirer.

## II. Suppression des Colonnes Non Pertinentes

In [2]:
# Ces colonnes n'apportent aucune information prédictive
cols_to_drop = ['index', 'dateCrawled', 'dateCreated', 'lastSeen', 
                'nrOfPictures', 'postalCode', 'abtest', 'seller', 'offerType']

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print(f"Colonnes supprimées. Shape : {df.shape}")
print(f"Colonnes restantes : {df.columns.tolist()}")

Colonnes supprimées. Shape : (371528, 12)
Colonnes restantes : ['name', 'price', 'vehicleType', 'yearOfRegistration', 'gearbox', 'powerPS', 'model', 'kilometer', 'monthOfRegistration', 'fuelType', 'brand', 'notRepairedDamage']


### Interprétation

Nous supprimons :
- **Identifiants techniques** (`index`) : sans valeur prédictive.
- **Colonnes constantes ou quasi-constantes** (`nrOfPictures`, `seller`, `offerType`) : leur variance est quasi nulle, elles n'apportent donc aucun pouvoir discriminant.
- **Colonnes temporelles de scraping** (`dateCrawled`, `dateCreated`, `lastSeen`) : elles décrivent le moment où l'annonce a été collectée, pas une caractéristique du véhicule.
- **`postalCode`** : trop granulaire (des milliers de valeurs uniques) pour être exploitable sans encodage géographique avancé, et redondant avec d'autres variables.
- **`abtest`** : variable liée à un test A/B du site d'annonces, sans lien avec le prix réel du véhicule.

Cette réduction de dimensionnalité limite le bruit et la complexité du modèle sans perte d'information utile, comme identifié lors de l'EDA.

## III. Filtrage des Valeurs Aberrantes

In [3]:
import numpy as np

print("=" * 55)
print("FILTRAGE STATISTIQUE PAR METHODE IQR")
print("=" * 55)
shape_avant = df.shape[0]
print(f"Lignes avant filtrage : {shape_avant:,}")

# ETAPE 1 — VARIABLE CIBLE price
# IQR applique sur log(price) car distribution asymetrique
log_price = np.log1p(df['price'])
Q1_p  = log_price.quantile(0.25)
Q3_p  = log_price.quantile(0.75)
IQR_p = Q3_p - Q1_p
low_p  = np.expm1(Q1_p - 1.5 * IQR_p)
high_p = np.expm1(Q3_p + 1.5 * IQR_p)
low_p  = max(low_p, 100)
df = df[(df['price'] >= low_p) & (df['price'] <= high_p)]
print(f"\nETAPE 1 - price (IQR sur log) :")
print(f"  [{low_p:,.0f} € → {high_p:,.0f} €] | Restantes : {df.shape[0]:,}")

# ETAPE 2a — FEATURE yearOfRegistration
Q1_y  = df['yearOfRegistration'].quantile(0.25)
Q3_y  = df['yearOfRegistration'].quantile(0.75)
IQR_y = Q3_y - Q1_y
low_y  = Q1_y - 1.5 * IQR_y
high_y = Q3_y + 1.5 * IQR_y
df = df[(df['yearOfRegistration'] >= low_y) & (df['yearOfRegistration'] <= high_y)]
print(f"\nETAPE 2a - yearOfRegistration (IQR) :")
print(f"  [{low_y:.0f} → {high_y:.0f}] | Restantes : {df.shape[0]:,}")

# ETAPE 2b — FEATURE powerPS
Q1_ps  = df['powerPS'].quantile(0.25)
Q3_ps  = df['powerPS'].quantile(0.75)
IQR_ps = Q3_ps - Q1_ps
low_ps  = max(Q1_ps - 1.5 * IQR_ps, 10)
high_ps = Q3_ps + 1.5 * IQR_ps
df = df[(df['powerPS'] >= low_ps) & (df['powerPS'] <= high_ps)]
print(f"\nETAPE 2b - powerPS (IQR) :")
print(f"  [{low_ps:.0f} → {high_ps:.0f} PS] | Restantes : {df.shape[0]:,}")

# ETAPE 2c — FEATURE kilometer
Q1_km  = df['kilometer'].quantile(0.25)
Q3_km  = df['kilometer'].quantile(0.75)
IQR_km = Q3_km - Q1_km
low_km  = max(Q1_km - 1.5 * IQR_km, 0)
high_km = Q3_km + 1.5 * IQR_km
df = df[(df['kilometer'] >= low_km) & (df['kilometer'] <= high_km)]
print(f"\nETAPE 2c - kilometer (IQR) :")
print(f"  [{low_km:,.0f} → {high_km:,.0f} km] | Restantes : {df.shape[0]:,}")

df.reset_index(drop=True, inplace=True)
print(f"\n{'='*55}")
print(f"Lignes avant  : {shape_avant:,}")
print(f"Lignes apres  : {df.shape[0]:,}")
print(f"Supprimees    : {shape_avant - df.shape[0]:,} ({(shape_avant - df.shape[0])/shape_avant*100:.1f}%)")
print(f"\nPlage de fonctionnement du modele :")
print(f"  Prix      : {df['price'].min():,.0f} € → {df['price'].max():,.0f} €")
print(f"  Annee     : {df['yearOfRegistration'].min():.0f} → {df['yearOfRegistration'].max():.0f}")
print(f"  Puissance : {df['powerPS'].min():.0f} → {df['powerPS'].max():.0f} PS")
print(f"  Km        : {df['kilometer'].min():,.0f} → {df['kilometer'].max():,.0f} km")

FILTRAGE STATISTIQUE PAR METHODE IQR
Lignes avant filtrage : 371,528



ETAPE 1 - price (IQR sur log) :
  [100 € → 112,685 €] | Restantes : 357,862



ETAPE 2a - yearOfRegistration (IQR) :
  [1986 → 2022] | Restantes : 351,526

ETAPE 2b - powerPS (IQR) :
  [10 → 262 PS] | Restantes : 306,423

ETAPE 2c - kilometer (IQR) :
  [87,500 → 187,500 km] | Restantes : 254,902

Lignes avant  : 371,528
Lignes apres  : 254,902
Supprimees    : 116,626 (31.4%)

Plage de fonctionnement du modele :
  Prix      : 100 € → 111,111 €


  Annee     : 1986 → 2019
  Puissance : 10 → 262 PS
  Km        : 90,000 → 150,000 km


### Interprétation

Les seuils appliqués sont directement issus des observations de l'EDA (distributions très asymétriques et présence de valeurs extrêmes/incohérentes) :
- **`price`** : on conserve les annonces entre 100 € et 100 000 €. Les prix proches de 0 correspondent souvent à des annonces erronées ou à des pièces détachées, et les prix extrêmes (> 100 000 €) sont des cas rares qui déséquilibreraient l'apprentissage.
- **`yearOfRegistration`** : on garde les véhicules immatriculés entre 1950 et 2024. Le dataset contient des valeurs aberrantes comme `1000` ou `9999`, impossibles pour un véhicule en circulation.
- **`powerPS`** : on conserve une puissance comprise entre 10 et 500 ch. Les valeurs à 0 ch sont des données manquantes déguisées, et au-delà de 500 ch on entre dans des catégories de véhicules de niche non représentatives.
- **`kilometer`** : on plafonne à 200 000 km, valeur cohérente avec le maximum déclaratif du site d'annonces.

Ce filtrage supprime les enregistrements les plus bruités tout en conservant la grande majorité des données, ce qui rend la distribution de la cible plus exploitable pour la modélisation.

## IV. Traitement des Valeurs Manquantes

In [4]:
print("=== Valeurs manquantes avant imputation ===")
missing = df.isnull().sum()
print(missing[missing > 0])

=== Valeurs manquantes avant imputation ===


vehicleType          18458
gearbox               4910
model                 9785
fuelType             17047
notRepairedDamage    40893
dtype: int64


In [5]:
print("=== SUPPRESSION DES LIGNES AVEC VALEURS MANQUANTES ===")

cols_with_na = ['vehicleType', 'gearbox', 'fuelType', 
                'notRepairedDamage', 'model']

shape_avant = df.shape[0]

print(f"Lignes avant suppression : {shape_avant:,}")
print("\nValeurs manquantes par colonne :")
for col in cols_with_na:
    n_missing = df[col].isnull().sum()
    pct = n_missing / shape_avant * 100
    print(f"  {col:20s} : {n_missing:>7,} ({pct:.2f}%)")

# Suppression de toutes les lignes ayant au moins un NaN 
# dans ces colonnes
df.dropna(subset=cols_with_na, inplace=True)
df.reset_index(drop=True, inplace=True)

shape_apres = df.shape[0]
print(f"\nLignes apres suppression : {shape_apres:,}")
print(f"Lignes supprimees         : {shape_avant - shape_apres:,} "
      f"({(shape_avant - shape_apres)/shape_avant*100:.1f}%)")

# Verification finale
print("\n=== VERIFICATION ===")
remaining_na = df[cols_with_na].isnull().sum().sum()
print(f"Valeurs manquantes restantes : {remaining_na}")

=== SUPPRESSION DES LIGNES AVEC VALEURS MANQUANTES ===
Lignes avant suppression : 254,902

Valeurs manquantes par colonne :
  vehicleType          :  18,458 (7.24%)
  gearbox              :   4,910 (1.93%)
  fuelType             :  17,047 (6.69%)
  notRepairedDamage    :  40,893 (16.04%)
  model                :   9,785 (3.84%)



Lignes apres suppression : 191,873
Lignes supprimees         : 63,029 (24.7%)

=== VERIFICATION ===
Valeurs manquantes restantes : 0


### Interprétation

Les valeurs manquantes concernent uniquement des **variables catégorielles** (`vehicleType`, `gearbox`, `fuelType`, `notRepairedDamage`, `model`). Aucune variable numérique n'est concernée après le filtrage de la section précédente.

Nous choisissons la **suppression des lignes** contenant au moins une valeur manquante sur ces colonnes, plutôt que :
- l'**imputation par le mode**, qui introduirait artificiellement une sur-représentation de la modalité la plus fréquente et pourrait biaiser l'apprentissage du modèle ;
- la création d'une catégorie `"manquant"`, qui aurait pu être pertinente mais complexifierait l'analyse de l'importance des variables sans apporter de signal clair ici.

Le dataset initial (254,902 lignes après filtrage IQR) est suffisamment volumineux pour que la suppression de ces lignes (environ un quart des données) ne compromette pas la qualité de l'entraînement, tout en garantissant que les données restantes sont entièrement réelles et non biaisées par une imputation artificielle. La vérification finale confirme qu'il ne reste plus aucune valeur manquante dans le dataset.

## V. Feature Engineering

In [6]:
# Création de la variable âge du véhicule
df['car_age'] = 2016 - df['yearOfRegistration']
print("Variable 'car_age' créée (2024 - yearOfRegistration)")

# Transformation logarithmique de la variable cible
df['log_price'] = np.log1p(df['price'])
print("Variable 'log_price' créée (log(1 + price))")

# Vérification
print("\nAperçu des nouvelles variables :")
print(df[['yearOfRegistration', 'car_age', 'price', 'log_price']].head())

Variable 'car_age' créée (2024 - yearOfRegistration)
Variable 'log_price' créée (log(1 + price))

Aperçu des nouvelles variables :
   yearOfRegistration  car_age  price  log_price
0                2001       15   1500   7.313887
1                2008        8   3600   8.188967
2                1995       21    650   6.478510
3                2004       12   2200   7.696667
4                2004       12   2000   7.601402


### Interprétation

Deux nouvelles variables sont dérivées des données existantes :

- **`car_age`** (âge du véhicule) : transformation de `yearOfRegistration` en une grandeur plus directement interprétable et plus stable dans le temps. Une « année de référence » fixe (2024) évite que le modèle n'apprenne une relation purement chronologique qui deviendrait obsolète, et l'âge a généralement une relation plus monotone et intuitive avec la décote du véhicule.

- **`log_price`** (log du prix) : l'EDA a montré que `price` est fortement asymétrique à droite (quelques annonces très chères tirent la distribution). La transformation `log(1 + price)` rapproche la distribution de la cible d'une loi normale, ce qui est bénéfique pour les modèles linéaires (Régression Linéaire, Ridge/Lasso) et stabilise la variance des résidus. Elle servira de **variable cible pour l'entraînement**, et il faudra appliquer la transformation inverse (`expm1`) pour revenir à un prix en euros lors de l'évaluation finale.

## VI. Encodage des Variables Catégorielles

In [7]:
# Affichage des variables catégorielles à encoder
cat_to_encode = ['vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage', 'brand', 'model']
print("Variables catégorielles à encoder :")
for col in cat_to_encode:
    print(f"  {col} : {df[col].nunique()} valeurs uniques")

Variables catégorielles à encoder :
  vehicleType : 8 valeurs uniques
  gearbox : 2 valeurs uniques
  fuelType : 7 valeurs uniques
  notRepairedDamage : 2 valeurs uniques
  brand : 39 valeurs uniques
  model : 246 valeurs uniques


In [8]:
# Label Encoding pour toutes les variables catégorielles
encoders = {}

for col in cat_to_encode:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"{col} encodé -> {col}_encoded")

print("=== VERIFICATION APRES CORRECTION ===")
for col, enc in encoders.items():
    print(f"{col:25s} : {len(enc.classes_)} classes -> {list(enc.classes_)[:8]}")

vehicleType encodé -> vehicleType_encoded


gearbox encodé -> gearbox_encoded
fuelType encodé -> fuelType_encoded
notRepairedDamage encodé -> notRepairedDamage_encoded
brand encodé -> brand_encoded


model encodé -> model_encoded


=== VERIFICATION APRES CORRECTION ===
vehicleType               : 8 classes -> ['andere', 'bus', 'cabrio', 'coupe', 'kleinwagen', 'kombi', 'limousine', 'suv']
gearbox                   : 2 classes -> ['automatik', 'manuell']
fuelType                  : 7 classes -> ['andere', 'benzin', 'cng', 'diesel', 'elektro', 'hybrid', 'lpg']
notRepairedDamage         : 2 classes -> ['ja', 'nein']
brand                     : 39 classes -> ['alfa_romeo', 'audi', 'bmw', 'chevrolet', 'chrysler', 'citroen', 'dacia', 'daewoo']
model                     : 246 classes -> ['100', '145', '147', '156', '159', '1_reihe', '1er', '200']


### Interprétation

Les algorithmes de Machine Learning de scikit-learn nécessitent des entrées numériques : les variables catégorielles (`vehicleType`, `gearbox`, `fuelType`, `notRepairedDamage`, `brand`, `model`) doivent donc être converties.

Nous utilisons un **`LabelEncoder`**, qui attribue un entier unique à chaque modalité. Ce choix est pertinent ici car :
- certaines colonnes comme `model` et `brand` comptent un grand nombre de modalités (plusieurs centaines pour `model`) — un One-Hot Encoding exploserait la dimensionnalité du dataset ;
- les modèles à base d'arbres (Random Forest, Gradient Boosting, XGBoost, LightGBM) gèrent très bien les variables encodées en entiers, sans supposer d'ordre entre les catégories.

**Limite à garder en tête :** pour des modèles linéaires (Régression Linéaire, Ridge/Lasso) ou à base de distance (KNN, SVR), le `LabelEncoder` introduit un ordre artificiel entre les modalités qui n'a pas de sens métier. Si ces modèles sont retenus, un encodage alternatif (One-Hot, Target Encoding) pourra être envisagé en complément.

Les variables encodées sont stockées avec le suffixe `_encoded`, et les `encoders` sont conservés en mémoire pour pouvoir transformer de nouvelles données de la même manière (et être réutilisés en inférence).

## VII. Sélection des Features Finales

In [9]:
# Features sélectionnées pour la modélisation
features = [
    'car_age',
    'powerPS', 
    'kilometer',
    'vehicleType_encoded',
    'gearbox_encoded',
    'fuelType_encoded',
    'notRepairedDamage_encoded',
    'brand_encoded',
    'model_encoded'
]

target = 'log_price'

X = df[features]
y = df[target]

print(f"Features sélectionnées : {len(features)}")
print(f"Variable cible : {target}")
print(f"Shape X : {X.shape}")
print(f"Shape y : {y.shape}")
X.head()

Features sélectionnées : 9
Variable cible : log_price
Shape X : (191873, 9)
Shape y : (191873,)


,car_age,powerPS,kilometer,vehicleType_encoded,gearbox_encoded,fuelType_encoded,notRepairedDamage_encoded,brand_encoded,model_encoded
0,15,75,150000,4,1,1,1,37,116
1,8,69,90000,4,1,3,1,31,101
2,21,102,150000,6,1,1,0,2,11
3,12,109,150000,2,1,1,1,25,8
4,12,105,150000,6,1,1,1,19,10


### Interprétation

La matrice `X` rassemble :
- les **variables numériques** directement exploitables (`car_age`, `powerPS`, `kilometer`) ;
- les **variables catégorielles encodées** (`*_encoded`), qui remplacent leurs versions textuelles d'origine.

Les colonnes textuelles brutes (`vehicleType`, `brand`, `model`, ...), `price`, `yearOfRegistration` et `monthOfRegistration` sont volontairement exclues de `X` :
- `price` et `yearOfRegistration` sont redondantes avec `log_price` et `car_age` (la cible elle-même, ou une variable dérivée directement liée) — les conserver provoquerait une fuite de données (*data leakage*) ;
- `monthOfRegistration` apporte un signal négligeable et complique le formulaire de prédiction en production, elle est donc retirée des features finales ;
- les versions textuelles des variables catégorielles sont redondantes avec leurs versions encodées.

`y` correspond à `log_price`, la cible transformée définie en section V, choisie pour stabiliser la variance et l'asymétrie de la distribution du prix.

## VIII. Division Train / Test

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Données divisées avec succès :")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_train : {y_train.shape}")
print(f"   y_test  : {y_test.shape}")

Données divisées avec succès :
   X_train : (153498, 9)
   X_test  : (38375, 9)
   y_train : (153498,)
   y_test  : (38375,)


### Interprétation

Nous réservons **20 % des données pour le test** (`test_size=0.2`) et **80 % pour l'entraînement**, un compromis classique qui laisse suffisamment de données pour apprendre tout en gardant un échantillon de test représentatif pour évaluer la généralisation.

Le paramètre **`random_state=42`** fixe la graine aléatoire du tirage : il garantit que la division est **reproductible** d'une exécution à l'autre, ce qui est indispensable pour comparer équitablement plusieurs modèles dans le notebook suivant.

Cette séparation est effectuée **avant** la normalisation (section IX) afin d'éviter toute fuite d'information de l'ensemble de test vers l'ensemble d'entraînement.

## IX. Normalisation des Features Numériques

In [11]:
from sklearn.preprocessing import StandardScaler

num_features = ['car_age', 'powerPS', 'kilometer']

scaler = StandardScaler()
scaler.fit(X_train[num_features])  # fit uniquement — la transformation est geree par le pipeline

# Les lignes ci-dessous sont commentees volontairement :
# le pipeline_final (notebook 04) contient deja un StandardScaler.
# Sauvegarder des donnees deja scalees entrainerait un double scaling a la prediction.
# X_train[num_features] = scaler.fit_transform(X_train[num_features])
# X_test[num_features]  = scaler.transform(X_test[num_features])

print("StandardScaler ajuste sur X_train (fit uniquement, sans transformation).")
print("Les CSV seront sauvegardes non-scales — le pipeline applique le scaling a la prediction.")
for f in num_features:
    print(f"   -> {f}")

StandardScaler ajuste sur X_train (fit uniquement, sans transformation).
Les CSV seront sauvegardes non-scales — le pipeline applique le scaling a la prediction.
   -> car_age
   -> powerPS
   -> kilometer


### Interprétation

Le **`StandardScaler`** centre chaque variable numérique sur une moyenne de 0 et une variance de 1 (`(x - moyenne) / écart-type`). Cette étape est :

- **indispensable** pour les modèles sensibles à l'échelle des variables comme la **Régression Ridge/Lasso**, le **SVR** ou le **KNN**, qui reposent sur des distances ou des pénalités appliquées aux coefficients ;
- **sans effet négatif** sur les modèles à base d'arbres (Random Forest, XGBoost, LightGBM), qui sont invariants aux transformations monotones — appliquer le scaler ne pose donc aucun problème même si l'on teste plusieurs familles de modèles.

**Point méthodologique important :** le scaler est **ajusté (`fit`) uniquement sur `X_train`**, puis simplement **appliqué (`transform`) sur `X_test`**. Cela évite toute fuite d'information de l'ensemble de test vers l'ensemble d'entraînement et reproduit fidèlement les conditions réelles de déploiement, où les nouvelles données sont toujours inconnues au moment de l'entraînement.

Seules les variables numériques continues sont normalisées ; les variables catégorielles encodées ne le sont pas, car leurs valeurs entières représentent des identifiants de catégories et non des grandeurs continues.

## X. Sauvegarde des Données Preprocessées

In [12]:
import joblib

# Sauvegarde des données
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Sauvegarde du scaler et des encoders
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(encoders, '../models/encoders.pkl')
joblib.dump(features, '../models/features_list.pkl')

print("Données sauvegardées dans data/")
print("Scaler sauvegardé dans models/scaler.pkl")
print("Encoders sauvegardés dans models/encoders.pkl")

Données sauvegardées dans data/
Scaler sauvegardé dans models/scaler.pkl
Encoders sauvegardés dans models/encoders.pkl


### Interprétation

Nous sauvegardons à la fois **les données** et **les artefacts de transformation** :

- `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` : les jeux de données prêts pour la modélisation, afin que le notebook suivant (entraînement des modèles) puisse les recharger directement sans répéter tout le pipeline ;
- `scaler.pkl`, `encoders.pkl`, `features_list.pkl` : les objets ajustés sur les données d'entraînement (`StandardScaler`, `LabelEncoder`s, liste des features). Ils devront être **réutilisés tels quels** (sans réajustement) pour transformer toute nouvelle donnée à prédire — c'est ce qui garantit la cohérence entre l'entraînement et l'inférence en production.

Sérialiser ces objets avec `joblib` (plutôt que de les recalculer) assure la **reproductibilité** du pipeline complet : EDA -> preprocessing -> modélisation -> déploiement.

## XI. Synthèse du Preprocessing

### Récapitulatif des étapes effectuées

| Étape | Action | Justification |
|-------|--------|---------------|
| Suppression colonnes | 9 colonnes supprimées | Non pertinentes pour la prédiction |
| Filtrage outliers | price, yearOfRegistration, powerPS, kilometer | Valeurs aberrantes faussant le modèle |
| Valeurs manquantes | Suppression des lignes concernées | Evite le biais d'une imputation artificielle |
| Feature Engineering | car_age, log_price | Améliore la linéarité et la pertinence |
| Encodage | LabelEncoder sur 6 colonnes | Conversion numérique requise par sklearn |
| Sélection features | 9 features (monthOfRegistration exclue) | Simplifie le pipeline et le formulaire de prédiction |
| Normalisation | StandardScaler sur 3 features numériques | Nécessaire pour SVR, KNN, Ridge/Lasso |
| Split | 80% train / 20% test, random_state=42 | Reproductibilité et évaluation fiable |

Le dataset est désormais propre, complet et entièrement numérique : il est prêt à être utilisé dans le notebook suivant pour l'entraînement et la comparaison des modèles de régression.

In [13]:
import json

print("=== GENERATION DES CONTRAINTES PAR MODELE ===")

MIN_SAMPLES = 30
constraints = {}

for brand_val in df['brand'].unique():
    df_brand = df[df['brand'] == brand_val]
    
    # Bornes de la marque entiere (fallback)
    brand_ps_min = df_brand['powerPS'].min()
    brand_ps_max = df_brand['powerPS'].max()
    brand_fuels  = sorted(df_brand['fuelType'].unique().tolist())
    brand_gearbox = sorted(df_brand['gearbox'].unique().tolist())
    
    for model_val in df_brand['model'].unique():
        df_model = df_brand[df_brand['model'] == model_val]
        n = len(df_model)
        
        key = f"{brand_val}||{model_val}"
        
        if n >= MIN_SAMPLES:
            ps_min = int(df_model['powerPS'].min() * 0.9)
            ps_max = int(df_model['powerPS'].max() * 1.1)
            fuels  = sorted(df_model['fuelType'].unique().tolist())
            gearbox_opts = sorted(df_model['gearbox'].unique().tolist())
        else:
            # Fallback sur les bornes de la marque
            ps_min = int(brand_ps_min * 0.9)
            ps_max = int(brand_ps_max * 1.1)
            fuels  = brand_fuels
            gearbox_opts = brand_gearbox
        
        constraints[key] = {
            'powerPS_min' : max(10, ps_min),
            'powerPS_max' : min(500, ps_max),
            'fuelTypes'   : fuels,
            'gearboxes'   : gearbox_opts,
            'n_samples'   : int(n)
        }

print(f"Contraintes generees pour {len(constraints)} combinaisons brand+model")

# Exemple d'affichage
example_key = list(constraints.keys())[0]
print(f"\nExemple ({example_key}) :")
print(json.dumps(constraints[example_key], indent=2))

# Sauvegarde
joblib.dump(constraints, '../models/model_constraints.pkl')
print(f"\nContraintes sauvegardees : models/model_constraints.pkl")

=== GENERATION DES CONTRAINTES PAR MODELE ===


Contraintes generees pour 293 combinaisons brand+model

Exemple (volkswagen||golf) :
{
  "powerPS_min": 39,
  "powerPS_max": 286,
  "fuelTypes": [
    "andere",
    "benzin",
    "cng",
    "diesel",
    "elektro",
    "lpg"
  ],
  "gearboxes": [
    "automatik",
    "manuell"
  ],
  "n_samples": 16103
}

Contraintes sauvegardees : models/model_constraints.pkl
